In [3]:
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from crypto_panel import build_crypto_panel  # Achille: stock vs crypto panel
from Sem_investment_comparison import make_investment_comparison



# --------------------------------------------------
# Existing dashboard charts
# --------------------------------------------------

def make_charts(data, ticker, period):

    close = data["Close"].dropna()

    if close.empty:
        raise ValueError("No closing-price data was returned for this ticker.")

    df = pd.DataFrame(index=close.index)
    df["Close"] = close

    # Fetch 10-year Treasury yield
    tnx_data = yf.Ticker("^TNX").history(
        period=period,
        auto_adjust=False
    )

    if tnx_data.empty or tnx_data["Close"].dropna().empty:
        raise ValueError("No 10-year Treasury yield data was returned.")

    tnx_close = tnx_data["Close"].dropna().reindex(
        close.index,
        method="ffill"
    )

    df["TNX"] = tnx_close

    # Chart 1: Closing price and Treasury yield
    price_fig, price_ax = plt.subplots(figsize=(8, 3.2))

    price_ax.plot(
        close.index,
        close,
        label="Close",
        color="#176B87",
        linewidth=1.8
    )

    price_ax.set_title(f"{ticker} closing price")
    price_ax.set_ylabel("Price")

    tnx_ax = price_ax.twinx()

    tnx_ax.plot(
        close.index,
        tnx_close,
        label="10-year Treasury yield",
        color="#E07A3F",
        linewidth=1.3
    )

    tnx_ax.set_ylabel("Yield (%)")

    lines = price_ax.get_lines() + tnx_ax.get_lines()

    price_ax.legend(
        lines,
        [line.get_label() for line in lines],
        frameon=False
    )

    # Chart 2: Existing stock and Treasury scatter chart
    variables = ["Close", "TNX"]
    colors = ["#42A38A", "#E67E22"]

    volume_fig, volume_ax = plt.subplots(figsize=(8, 3.2))

    for var, color in zip(variables, colors):

        volume_ax.scatter(
            df.index,
            df[var].fillna(0),
            label=var,
            color=color,
            alpha=0.7
        )

    volume_ax.set_title(f"{ticker} trading volume")
    volume_ax.set_ylabel("Shares")
    volume_ax.legend(frameon=False)

    # Chart 3: Interactive five-investment comparison
    comparison_fig = make_investment_comparison(
        data,
        ticker,
        period
    )

    # Formatting for Matplotlib charts
    for axis in (price_ax, volume_ax):
        axis.grid(axis="y", alpha=0.22)
        axis.tick_params(axis="x", labelrotation=25)

    for figure in (price_fig, volume_fig):
        figure.tight_layout()

    return price_fig, volume_fig, comparison_fig


# --------------------------------------------------
# Error handling and placeholder charts
# --------------------------------------------------

def placeholder_plot(title, message):

    figure, axis = plt.subplots(figsize=(8, 3.2))

    axis.text(
        0.5,
        0.5,
        message,
        ha="center",
        va="center",
        color="#666666"
    )

    axis.set_title(title)
    axis.set_axis_off()
    figure.tight_layout()

    return figure


def update_dashboard(ticker, period):

    ticker = ticker.strip().upper()

    try:

        if not ticker:
            raise ValueError("Enter a ticker symbol, such as AAPL.")

        data = yf.Ticker(ticker).history(
            period=period,
            auto_adjust=False
        )

        if data.empty:
            raise ValueError(
                "No data was found. Check the ticker or try another period."
            )

        charts = make_charts(data, ticker, period)

        return (
            f"Loaded {len(data):,} trading days for {ticker}.",
            *charts
        )

    except Exception as error:

        message = str(error)

        charts = tuple(
            placeholder_plot(title, message)
            for title in (
                "Closing price",
                "Trading volume",
                "Investment comparison"
            )
        )

        return (
            f"Unable to load {ticker or 'ticker'}: {message}",
            *charts
        )


def initial_plot(title):
    return placeholder_plot(
        title,
        "Enter a ticker and select Load charts"
    )


# --------------------------------------------------
# Gradio dashboard
# --------------------------------------------------

demo = gr.Blocks(title="Market charts")

with demo:

    gr.Markdown(
        "# Market charts\n"
        "Explore stock prices, Treasury yields and investment performance."
    )

    with gr.Row():

        ticker_input = gr.Textbox(
            label="Ticker",
            value="AAPL",
            placeholder="AAPL"
        )

        period_input = gr.Dropdown(
            choices=["1mo", "3mo", "6mo", "1y", "2y", "5y"],
            value="1y",
            label="History"
        )

        load_button = gr.Button(
            "Load charts",
            variant="primary"
        )

    status = gr.Markdown(
        "Ready. Load a ticker to display its charts."
    )

    with gr.Row():

        with gr.Group():

            gr.Markdown("### Price")

            price_plot = gr.Plot(
                value=initial_plot("Closing price"),
                label="Closing price"
            )

        with gr.Group():

            gr.Markdown("### Treasury and Stock")

            volume_plot = gr.Plot(
                value=initial_plot("Trading volume"),
                label="Trading volume"
            )

    # Second row: New comparison and Achille's crypto panel
    with gr.Row():

        with gr.Group():

            gr.Markdown("### Investment Performance Comparison")

            vnq_plot = gr.Plot(
                value=initial_plot("Investment comparison"),
                label="Investment comparison"
            )

        # Achille's original crypto panel remains unchanged
        build_crypto_panel(
            ticker_input,
            period_input,
            load_button
        )

    load_button.click(
        fn=update_dashboard,
        inputs=[
            ticker_input,
            period_input
        ],
        outputs=[
            status,
            price_plot,
            volume_plot,
            vnq_plot
        ]
    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
